This notebook uploads page-level transcriptions from `04.CoMMa/00.data/02.csvs_arkindex/00.exact_page_nmb_match` and `04.CoMMa/00.data/02.csvs_arkindex/03.last_img_absent_in_CoMMA` to Arkindex.

The CSV filenames are Arkindex manuscript IDs. Each CSV row already contains the target page id in `arkindex_page_id` and the text to publish in `text`.

## 1. Environment

In [1]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path
import os
import time
import uuid

import pandas as pd
import requests

In [5]:
os.chdir("C:\\Users\\yatsyk\\Documents\\NextCloud_Synchro\\")

## 2. Configuration

The upload cell will only write to Arkindex when `DRY_RUN = False`.

In [43]:
ARKINDEX_BASE_URL = "https://demo.arkindex.org"
TOKEN_PATH = Path("TEKLIA/notebooks/token_API.txt")

CSV_DIRS = [
    Path("passim_BoH/04.CoMMa/00.data/02.csvs_arkindex/00.exact_page_nmb_match"),
    Path("passim_BoH/04.CoMMa/00.data/02.csvs_arkindex/03.last_img_absent_in_CoMMA"),
]
REPORT_DIR = Path("passim_BoH/04.CoMMa/02.metadata/arkindex_transcription_upload")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

WORKER_ID_PATH = Path("TEKLIA/notebooks/worker_id_import_CoMMA_transcriptions.txt")
WORKER_RUN_ID = TOKEN_PATH.read_text(encoding="utf-8").strip()

DRY_RUN = False
BATCH_SIZE = 50
ORIENTATION = "horizontal-lr"
CONFIDENCE = 0.5
SKIP_EMPTY_TEXT = True
SLEEP_BETWEEN_BATCHES_SECONDS = 0.1

CSV_DIRS, REPORT_DIR

([WindowsPath('passim_BoH/04.CoMMa/00.data/02.csvs_arkindex/00.exact_page_nmb_match'),
  WindowsPath('passim_BoH/04.CoMMa/00.data/02.csvs_arkindex/03.last_img_absent_in_CoMMA')],
 WindowsPath('passim_BoH/04.CoMMa/02.metadata/arkindex_transcription_upload'))

In [26]:
ARKINDEX_TOKEN = TOKEN_PATH.read_text(encoding="utf-8").strip()

headers = {
    "Authorization": f"Token {ARKINDEX_TOKEN}",
    "Accept": "application/json",
    "Content-Type": "application/json",
}


def arkindex_request(method: str, path: str, *, expected=(200,), **kwargs) -> requests.Response:
    url = f"{ARKINDEX_BASE_URL.rstrip('/')}{path}"
    response = requests.request(method, url, headers=headers, timeout=60, **kwargs)
    if response.status_code not in expected:
        print(f"{method} {url}")
        print(f"HTTP {response.status_code}")
        print(response.text[:4000])
        response.raise_for_status()
    return response


# Quick authentication check.
corpus_response = arkindex_request("GET", "/api/v1/corpus/", expected=(200,))
print(f"Connected to {ARKINDEX_BASE_URL}: {len(corpus_response.json())} corpora visible")

Connected to https://demo.arkindex.org: 25 corpora visible


## 3. Load and validate CSV files

In [28]:
REQUIRED_COLUMNS = {"text", "arkindex_page_id"}


def read_transcription_csv(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path, dtype=str).fillna("")
    missing = REQUIRED_COLUMNS - set(df.columns)
    if missing:
        raise ValueError(f"{csv_path} is missing required columns: {sorted(missing)}")

    df = df.copy()
    df["manuscript_arkindex_id"] = csv_path.stem
    df["csv_path"] = str(csv_path)
    df["csv_row"] = range(2, len(df) + 2)  # CSV line number, including header.
    df["text_clean"] = df["text"].astype(str).str.strip()
    df["arkindex_page_id"] = df["arkindex_page_id"].astype(str).str.strip()
    return df

csv_paths = sorted(
    path
    for csv_dir in CSV_DIRS
    for path in csv_dir.glob("*.csv")
)

if not csv_paths:
    raise FileNotFoundError(f"No CSV files found in {CSV_DIRS}")

all_rows = pd.concat([read_transcription_csv(path) for path in csv_paths], ignore_index=True)
print(f"Loaded {len(all_rows):,} rows from {len(csv_paths):,} CSV files")
all_rows.head()

Loaded 37,612 rows from 139 CSV files


,page_id,folio,text,page_ordering,arkindex_page_id,manuscript_arkindex_id,csv_path,csv_row,text_clean
0,f0-plat-superieur.xml,,,1,00705702-8ce8-4944-86f0-3bbdc5a56e93,02181622-2bcc-4b35-b124-63c701e25860,passim_BoH\04.CoMMa\00.data\02.csvs_arkindex\0...,2,
1,f1-contreplat-sup.xml,,,2,8ed85d35-30ad-49e7-8e00-48485dff3e95,02181622-2bcc-4b35-b124-63c701e25860,passim_BoH\04.CoMMa\00.data\02.csvs_arkindex\0...,3,
2,f2-1r.xml,1r,Uolume de IA Ueuillelo de euillen io s est blane,3,ea0b67ec-c697-4265-b1de-d511a609c79e,02181622-2bcc-4b35-b124-63c701e25860,passim_BoH\04.CoMMa\00.data\02.csvs_arkindex\0...,4,Uolume de IA Ueuillelo de euillen io s est blane
3,f3-1v.xml,1v,,4,ce5926b5-c8cf-4093-b45c-f24de40cf1b5,02181622-2bcc-4b35-b124-63c701e25860,passim_BoH\04.CoMMa\00.data\02.csvs_arkindex\0...,5,
4,f4-2r.xml,2r,,5,ed550a4b-4e27-4ec9-92e3-729194a674b8,02181622-2bcc-4b35-b124-63c701e25860,passim_BoH\04.CoMMa\00.data\02.csvs_arkindex\0...,6,


In [29]:
invalid_page_ids = all_rows[all_rows["arkindex_page_id"].eq("")].copy()
empty_text_rows = all_rows[all_rows["text_clean"].eq("")].copy()
duplicate_targets = all_rows[
    all_rows["arkindex_page_id"].ne("")
    & all_rows["text_clean"].ne("")
    & all_rows.duplicated("arkindex_page_id", keep=False)
].sort_values(["arkindex_page_id", "manuscript_arkindex_id", "csv_row"])

validation_summary = pd.DataFrame(
    [
        {"check": "csv_files", "count": len(csv_paths)},
        {"check": "all_rows", "count": len(all_rows)},
        {"check": "missing_arkindex_page_id", "count": len(invalid_page_ids)},
        {"check": "empty_text_rows", "count": len(empty_text_rows)},
        {"check": "duplicate_nonempty_page_targets", "count": len(duplicate_targets)},
    ]
)

validation_summary

,check,count
0,csv_files,139
1,all_rows,37612
2,missing_arkindex_page_id,0
3,empty_text_rows,3896
4,duplicate_nonempty_page_targets,0


In [30]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

invalid_page_ids.to_csv(REPORT_DIR / f"invalid_page_ids_{timestamp}.csv", index=False, encoding="utf-8-sig")
empty_text_rows.to_csv(REPORT_DIR / f"empty_text_rows_{timestamp}.csv", index=False, encoding="utf-8-sig")
duplicate_targets.to_csv(REPORT_DIR / f"duplicate_targets_{timestamp}.csv", index=False, encoding="utf-8-sig")

if len(invalid_page_ids):
    raise ValueError(f"Found {len(invalid_page_ids)} rows without arkindex_page_id. See report directory.")

if len(duplicate_targets):
    raise ValueError(f"Found {len(duplicate_targets)} duplicate non-empty page targets. See report directory.")

upload_df = all_rows[all_rows["arkindex_page_id"].ne("")].copy()
if SKIP_EMPTY_TEXT:
    upload_df = upload_df[upload_df["text_clean"].ne("")].copy()

upload_df = upload_df.reset_index(drop=True)
print(f"Prepared {len(upload_df):,} page transcriptions for upload")
upload_df[["manuscript_arkindex_id", "page_id", "folio", "page_ordering", "arkindex_page_id", "text_clean"]].head(10)

Prepared 33,716 page transcriptions for upload


,manuscript_arkindex_id,page_id,folio,page_ordering,arkindex_page_id,text_clean
0,02181622-2bcc-4b35-b124-63c701e25860,f2-1r.xml,1r,3,ea0b67ec-c697-4265-b1de-d511a609c79e,Uolume de IA Ueuillelo de euillen io s est blane
1,02181622-2bcc-4b35-b124-63c701e25860,f8-4r.xml,4r,9,e9054d3f-2f99-4556-b05a-0edcfc1b3ccc,Nprincipio erat uerbum: & uerbum erat apud dcn...
2,02181622-2bcc-4b35-b124-63c701e25860,f9-4v.xml,4v,10,5c382b87-59e8-4c6b-b8a0-b65005b4820f,gnouit. In propria uenit & sui eũ non receperu...
3,02181622-2bcc-4b35-b124-63c701e25860,f10-5r.xml,5r,11,3c6c15cc-fe14-4903-a9b5-d943b14d0e97,bona temporalia: ut non amittamus aeterna. Per...
4,02181622-2bcc-4b35-b124-63c701e25860,f11-5v.xml,5v,12,0f32e5d6-0354-48e9-bd9e-eb4f72f204cb,uocabis nomen eius iesum. Hic e rit magnus: & ...
5,02181622-2bcc-4b35-b124-63c701e25860,f12-6r.xml,6r,13,d4883179-14ab-4d14-97b5-86fe92e066f7,N illo tempo. re. Cum natꝰ esset iestis in bet...
6,02181622-2bcc-4b35-b124-63c701e25860,f13-6v.xml,6v,14,8e9e8ddf-1ed3-4c55-8d18-fb36ee3e6a13,gat populum meum israel.Tunc he rodes rex clam...
7,02181622-2bcc-4b35-b124-63c701e25860,f14-7r.xml,7r,15,beaf8c13-b526-4baf-894c-4ad307e0bd4f,N illo tempo. re.Recumbẽ tibus undecim disci. ...
8,02181622-2bcc-4b35-b124-63c701e25860,f15-7v.xml,7v,16,c5efcde2-ea64-4c7c-b7ed-f8c406d7847a,Cua maror nolties.hristi lecuno im ighaan manu...
9,02181622-2bcc-4b35-b124-63c701e25860,f17-8v.xml,8v,18,f4eec0e3-02b1-4c55-8dc9-15dfae9c3f77,Cressus est dominus iesus cum discipulis suis ...


## 4. Build Arkindex payload

In [31]:
def build_transcription_item(row: pd.Series) -> dict:
    return {
        "element_id": row["arkindex_page_id"],
        "text": row["text_clean"],
        "confidence": CONFIDENCE,
        "orientation": ORIENTATION,
    }


payload_items = [build_transcription_item(row) for _, row in upload_df.iterrows()]
payload_preview = {
    "worker_run_id": WORKER_RUN_ID,
    "transcriptions": payload_items[:3],
}
payload_preview

{'worker_run_id': 'c59f8baa-5c7e-4316-9a77-e3b3f8de1201',
 'transcriptions': [{'element_id': 'ea0b67ec-c697-4265-b1de-d511a609c79e',
   'text': 'Uolume de IA Ueuillelo de euillen io s est blane',
   'confidence': 0.5,
   'orientation': 'horizontal-lr'},
  {'element_id': 'e9054d3f-2f99-4556-b05a-0edcfc1b3ccc',
   'text': 'Nprincipio erat uerbum: & uerbum erat apud dcnĩ: et deus erat uerbum. Hoc erat in principio apud deum.Om. nia per ipsum facta sunt: & sine ipso factum est nichil.Quod factum est in ipso uita erat & uita erat lux ho. minum: & lux in tenebris locet. et te nebre eam non comprehenderunt. Fuit homo missus a deo: cui nomen erat iohannes.Nic uenit in testimo. nium: ut testimonium perhiberet de lumine:ut omnes crederent per illũ. Non erat ille lux: sed ut testimoniũ perhiberet de lumine. Erat lux uera que illuminat omnem hominem uenientem in hunc mundum. In mundo erat: & mundus peripsum factus est: & mundus eum non co¬',
   'confidence': 0.5,
   'orientation': 'horizontal-lr'}

## 5. Upload

In [37]:
def iter_batches(items: list[dict], size: int):
    for start in range(0, len(items), size):
        yield start, items[start : start + size]


def create_transcriptions_bulk(batch_items: list[dict]) -> dict:
    payload = {
        "worker_run_id": WORKER_RUN_ID,
        "transcriptions": batch_items,
    }
    response = arkindex_request("POST", "/api/v1/transcription/bulk/", json=payload, expected=(201,))
    return response.json()


upload_log = []
preview_path = REPORT_DIR / f"payload_preview_{timestamp}.csv"
upload_log_path = REPORT_DIR / f"upload_log_{timestamp}.csv"

upload_df[["manuscript_arkindex_id", "csv_path", "csv_row", "arkindex_page_id", "text_clean"]].to_csv(
    preview_path,
    index=False,
    encoding="utf-8-sig",
)

upload_df

,page_id,folio,text,page_ordering,arkindex_page_id,manuscript_arkindex_id,csv_path,csv_row,text_clean
0,f2-1r.xml,1r,Uolume de IA Ueuillelo de euillen io s est blane,3,ea0b67ec-c697-4265-b1de-d511a609c79e,02181622-2bcc-4b35-b124-63c701e25860,passim_BoH\04.CoMMa\00.data\02.csvs_arkindex\0...,4,Uolume de IA Ueuillelo de euillen io s est blane
1,f8-4r.xml,4r,Nprincipio erat uerbum: & uerbum erat apud dcn...,9,e9054d3f-2f99-4556-b05a-0edcfc1b3ccc,02181622-2bcc-4b35-b124-63c701e25860,passim_BoH\04.CoMMa\00.data\02.csvs_arkindex\0...,10,Nprincipio erat uerbum: & uerbum erat apud dcn...
2,f9-4v.xml,4v,gnouit. In propria uenit & sui eũ non receperu...,10,5c382b87-59e8-4c6b-b8a0-b65005b4820f,02181622-2bcc-4b35-b124-63c701e25860,passim_BoH\04.CoMMa\00.data\02.csvs_arkindex\0...,11,gnouit. In propria uenit & sui eũ non receperu...
3,f10-5r.xml,5r,bona temporalia: ut non amittamus aeterna. Per...,11,3c6c15cc-fe14-4903-a9b5-d943b14d0e97,02181622-2bcc-4b35-b124-63c701e25860,passim_BoH\04.CoMMa\00.data\02.csvs_arkindex\0...,12,bona temporalia: ut non amittamus aeterna. Per...
4,f11-5v.xml,5v,uocabis nomen eius iesum. Hic e rit magnus: & ...,12,0f32e5d6-0354-48e9-bd9e-eb4f72f204cb,02181622-2bcc-4b35-b124-63c701e25860,passim_BoH\04.CoMMa\00.data\02.csvs_arkindex\0...,13,uocabis nomen eius iesum. Hic e rit magnus: & ...
...,...,...,...,...,...,...,...,...,...
33711,f409-200v.xml,200v,introire. Pr̃ nr̃ aue maria Dñe ihũ xp̃e adoro...,410,fb5ad063-e7c9-477a-91de-ff68ab44b74f,f35abe24-e8d7-4712-a1c6-156bcd35953c,passim_BoH\04.CoMMa\00.data\02.csvs_arkindex\0...,411,introire. Pr̃ nr̃ aue maria Dñe ihũ xp̃e adoro...
33712,f410-201r.xml,201r,Deuota contemp hctio heate marie uirginis iuxi...,411,273a5bea-8836-4d5e-9a69-168febd64dc9,f35abe24-e8d7-4712-a1c6-156bcd35953c,passim_BoH\04.CoMMa\00.data\02.csvs_arkindex\0...,412,Deuota contemp hctio heate marie uirginis iuxi...
33713,f411-201v.xml,201v,Ei tremebat cum Gnebni nuti penus ⁊ ncliti. Qu...,412,1efbb796-6c31-491d-ab32-970738e297b0,f35abe24-e8d7-4712-a1c6-156bcd35953c,passim_BoH\04.CoMMa\00.data\02.csvs_arkindex\0...,413,Ei tremebat cum Gnebni nuti penus ⁊ ncliti. Qu...
33714,f412-202r.xml,202r,Iuc ut urdeu cormeum in am̃ado christũ acum St...,413,aa55cf32-b556-40a7-a48e-edf429c8773e,f35abe24-e8d7-4712-a1c6-156bcd35953c,passim_BoH\04.CoMMa\00.data\02.csvs_arkindex\0...,414,Iuc ut urdeu cormeum in am̃ado christũ acum St...


In [ ]:
if DRY_RUN:
    print(f"DRY_RUN=True: no data was uploaded. Preview written to {preview_path}")

    total_batches = (len(payload_items) + BATCH_SIZE - 1) // BATCH_SIZE
    for batch_index, (start, batch_items) in enumerate(iter_batches(payload_items, BATCH_SIZE), start=1):
        source_slice = upload_df.iloc[start : start + len(batch_items)]
        try:
            result = create_transcriptions_bulk(batch_items)
            created = result.get("transcriptions", result if isinstance(result, list) else [])
            status = "OK"
            error = ""
        except Exception as exc:
            created = []
            status = "ERROR"
            error = repr(exc)

        upload_log.append(
            {
                "batch_index": batch_index,
                "batch_start": start,
                "batch_size": len(batch_items),
                "status": status,
                "created_count": len(created),
                "first_page_id": source_slice.iloc[0]["arkindex_page_id"],
                "last_page_id": source_slice.iloc[-1]["arkindex_page_id"],
                "error": error,
            }
        )
        pd.DataFrame(upload_log).to_csv(upload_log_path, index=False, encoding="utf-8-sig")

        print(f"Batch {batch_index:,}/{total_batches:,}: {status}, {len(created):,} created")
        if status == "ERROR":
            raise RuntimeError(f"Upload stopped on batch {batch_index}. See {upload_log_path}")

        time.sleep(SLEEP_BETWEEN_BATCHES_SECONDS)

    print(f"Upload complete. Log written to {upload_log_path}")

DRY_RUN=True: no data was uploaded. Preview written to passim_BoH\04.CoMMa\02.metadata\arkindex_transcription_upload\payload_preview_20260518_133702.csv
